# 🎲 Advanced AI Gamemaster - OpenEnv RL Training with Unsloth

This notebook implements **Multi-Dimensional Reward GRPO** to train a self-improving Gamemaster.

### Advanced RL Strategies Implemented:
1. **Weighted Reward Shaping:** We prioritize Rule Accuracy (60%) while still rewarding Narrative Consistency (20%) and Progression (20%).
2. **Group Relative Policy Optimization:** We sample 8 completions per turn to allow the model to compare successful vs. unsuccessful rule enforcement.
3. **OpenEnv State Persistence:** The agent is trained against a live Infinite Dungeon engine.

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install openenv-core pydantic

In [ ]:
from unsloth import FastLanguageModel, PatchFastRL
PatchFastRL("GRPO", FastLanguageModel)

from trl import GRPOTrainer, GRPOConfig
import asyncio
import re
import json
from client import GamemasterEnv
from models import GamemasterAction

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-1.5B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=16,
    use_gradient_checkpointing="unsloth",
)

### Multi-Dimensional Reward Functions
We define separate functions for each dimension. TRL will average these.

In [ ]:
ENV_URL = "https://Pruthvi1762-gamemaster-env.hf.space"

def get_env_result(generated_text):
    try:
        json_match = re.search(r'\{.*\}', generated_text, re.DOTALL)
        if not json_match: return None
        action_data = json.loads(json_match.group(0))
        action = GamemasterAction(**action_data)
        
        with GamemasterEnv(base_url=ENV_URL).sync() as client:
            client.reset()
            return client.step(action)
    except: return None

def rule_accuracy_reward(prompts, completions, **kwargs):
    return [get_env_result(c[0]["content"]).observation.metadata.get("rule_accuracy", -1.0) 
            if get_env_result(c[0]["content"]) else -2.0 for c in completions]

def progression_reward(prompts, completions, **kwargs):
    return [get_env_result(c[0]["content"]).observation.metadata.get("progression", 0.0) 
            if get_env_result(c[0]["content"]) else 0.0 for c in completions]

def narrative_reward(prompts, completions, **kwargs):
    return [get_env_result(c[0]["content"]).observation.metadata.get("narrative_quality", 0.0) 
            if get_env_result(c[0]["content"]) else 0.0 for c in completions]

In [ ]:
training_args = GRPOConfig(
    learning_rate = 5e-6,
    num_generations = 8, 
    max_completion_length = 512,
    output_dir = "outputs",
)

trainer = GRPOTrainer(
    model = model,
    reward_funcs = [
        rule_accuracy_reward,
        progression_reward,
        narrative_reward
    ],
    args = training_args,
    train_dataset = train_dataset, # (Assume dataset is loaded as in previous version)
)

trainer.train()